# 03 — Eval

Phase 3 (Per-Field-Accuracy Baseline), Phase 4 (Iterations-Auswertung + Synthese-Tabelle), Phase 6 (Skalierung 7B + 3B-Halluzinations-Klassen).

Predictions kommen aus `02_extract.ipynb`. Gold ist eure `annotation/meine_gold.csv` aus Phase 2.

## Run-Header

| Feld | Wert |
|---|---|
| Datum | _YYYY-MM-DD_ |
| Gold-Datei | `annotation/meine_gold.csv` |
| Aktive Predictions-Datei(en) | _ |
| Match-Entscheidung `skills_top3` | _ (Set / geordnete Liste) |
| Match-Entscheidung `gehalt_min_eur` | _ (exakt / Toleranz X %) |
| JSON-Parse-Fails (Anzahl) | _ |

## Phase 3 — Baseline-Accuracy auf 12 Hand-Gold-Anzeigen

Hypothese-Cell *vor* der Eval: welches Feld haltet ihr für am stärksten / am schwächsten — und warum?

# Evaluation - Baseline (Block 3.2)

| Feld | Wert |
|---|---|
| Datum | 2026-05-18 |
| Predictions-Datei | `daten/predictions_baseline.jsonl` |
| Gold-Datei | `AUFGABEN/annotation/meine_gold.csv` |

## Meine Hypothese (vor der Messung)
* **Stärkstes Feld:** Ich vermute, dass das Gehaltsangaben am besten abschneiden, weil Angaben zu den Gehältern meistens klar sind.
* **Schwächstes Feld:** Ich vermute, dass Angaben zu Homeoffice am schlechtesten abschneidet, weil sich die Varianten und das Verständnis von Home Office sich in der Arbeitswelt und von Anzeige zu Anzeige stark unterscheidet; selbst für Menschen. Erfahrungslevel lässt ebenfalls sehr viel Spielraum für unterschiedliche Interpretationen.

In [3]:
import pandas as pd
import json
from pathlib import Path

basispfad = Path("/home/jovyan/work/notebooks/LLM-Workshop/llm-workshop")
gold_pfad = basispfad / "AUFGABEN" / "annotation" / "meine_gold.csv"
predictions_pfad = basispfad / "daten" / "predictions_baseline.jsonl"

# Daten laden & säubern
gold_df = pd.read_csv(gold_pfad, encoding="utf-8-sig", dtype=str)
gold_df.columns = gold_df.columns.str.strip()
gold_df = gold_df.rename(columns={"id": "refnr"}) if "id" in gold_df.columns else gold_df.rename(columns={gold_df.columns[0]: "refnr"})
gold_df["refnr"] = gold_df["refnr"].str.strip()

preds_df = pd.DataFrame([json.loads(z) for z in open(predictions_pfad, "r", encoding="utf-8")])
preds_df["refnr"] = preds_df["refnr"].astype(str).str.strip()

eval_df = gold_df.merge(preds_df, on="refnr", how="inner")
print(f"Erfolgreich gejoint: {len(eval_df)} Anzeigen. Parse-Fails: {eval_df['extracted'].isna().sum()}\n")

# Accuracy Berechnungs-Schleife
felder = ["homeoffice", "vertragsart", "erfahrungslevel", "gehalt_min_eur", "gehalt_zeitraum", "skills_top3"]
ergebnisse = []

for feld in felder:
    n_korrekt = 0
    for _, row in eval_df.iterrows():
        if row["extracted"] is None: continue
        w_gold = str(row[feld]).strip().lower() if pd.notna(row[feld]) else ""
        w_pred = row["extracted"].get(feld)
        
        if feld == "skills_top3":
            g_set = set([s.strip() for s in w_gold.split("|") if s.strip()])
            p_set = set([str(s).strip().lower() for s in w_pred]) if isinstance(w_pred, list) else set()
            if g_set == p_set: n_korrekt += 1
        elif "gehalt" in feld:
            if w_gold in ["nan", "", "none"]: w_gold = None
            else: 
                try: w_gold = float(w_gold)
                except ValueError: pass
            if w_gold == w_pred: n_korrekt += 1
        else:
            w_pred_str = str(w_pred).strip().lower() if w_pred is not None else "none"
            if w_gold == w_pred_str: n_korrekt += 1
            
    ergebnisse.append({
        "Feld": feld, 
        "Accuracy (%)": round((n_korrekt / len(eval_df)) * 100, 1), 
        "n_korrekt": n_korrekt, 
        "n_total": len(eval_df)
    })

ergebnis_df = pd.DataFrame(ergebnisse)

ergebnis_df

Erfolgreich gejoint: 12 Anzeigen. Parse-Fails: 0



,Feld,Accuracy (%),n_korrekt,n_total
0,homeoffice,41.7,5,12
1,vertragsart,66.7,8,12
2,erfahrungslevel,33.3,4,12
3,gehalt_min_eur,16.7,2,12
4,gehalt_zeitraum,83.3,10,12
5,skills_top3,0.0,0,12


### Erläuterung der Evaluations-Ergebnisse (Baseline)

Diese Übersicht erklärt verständlich, wie die Metriken berechnet werden, welche "Spielregeln" für die KI gelten und was der Code im Hintergrund genau prüft.

---

#### 1. Die Rolle deiner CSV: Der "Gold-Standard"
In der KI-Entwicklung gilt die menschliche Einschätzung als die absolute Wahrheit (**Ground Truth** oder **Gold-Standard**). Das Modell wird nicht an einer universellen Wahrheit gemessen, sondern ausschließlich daran, **wie gut es dich imitiert**. Wenn das Modell eine Zahl findet, du das Feld aber als leer (`null`) definiert hast, gilt die KI-Antwort als **falsch**.

---

#### 2. Was bedeuten die Spaltenüberschriften?

| Spaltenname | Bedeutung | Erklärung für dein Ergebnis |
| :--- | :--- | :--- |
| **Feld** | Das untersuchte Schema-Feld | z. B. `homeoffice`, `skills_top3` etc. |
| **n_total** | Gesamtzahl der geprüften Anzeigen | Bei dir immer **12**, da deine Gold-Datei 12 Anzeigen enthält. |
| **n_korrekt** | Anzahl der exakten Übereinstimmungen | Wie oft waren du und die KI euch völlig einig. |
| **Accuracy (%)** | Die Trefferquote in Prozent | Formel: $(n\_korrekt / 12) \times 100$. Jede richtige Anzeige bringt $\sim 8,3\%$. |

---

#### 3. Die Spielregeln für "Korrektheit" (`n_korrekt`)
Der Computer ist beim Vergleich extrem strikt. Ein Punkt für `n_korrekt` wird nur vergeben, wenn folgende Bedingungen erfüllt sind:

* **Textfelder (`homeoffice`, `vertragsart`, `erfahrungslevel`):** * *Regel:* Striktes Text-Matching. Groß-/Kleinschreibung und Leerzeichen am Rand werden ignoriert. 
    * *Beispiel:* Sagst du `teilweise` und die KI `ja`, gibt es **0 Punkte**. Sagt die KI ebenfalls `teilweise`, gibt es **1 Punkt**.
* **Gehaltsfelder (`gehalt_min_eur`, `gehalt_zeitraum`):** * *Regel:* Exakter Zahlen- und Wertvergleich. 
    * *Beispiel:* Haben beide das Feld leer gelassen (`null`), ist das **korrekt**. Trägst du `50000` ein und die KI liest `49000` aus, ist es **falsch**.
* **Die Skill-Liste (`skills_top3`):** * *Regel:* Sogenanntes **Set-Match**. Die Reihenfolge der Skills ist egal, aber der Inhalt muss *haargenau* übereinstimmen.
    * *Warum hier 0.0% stehen:* Wenn du `['excel', 'powerpoint']` vorgibst, die KI aber `['ms excel', 'powerpoint']` schreibt, gilt das als **falsch**. Die KI lag inhaltlich oft richtig, hat aber deine genaue Schreibweise verfehlt.

---

#### 4. Grober Ablauf: Was macht der Code im Hintergrund?

1.  **Laden (Data Ingestion):** Der Code liest deine Einschätzungen (`meine_gold.csv`) und die Antworten der KI (`predictions_baseline.jsonl`) ein.
2.  **Verheiraten (Data Join):** Über die eindeutige ID (`refnr`) sorgt der Code dafür, dass für jede Anzeige die exakt passenden Zeilen miteinander verglichen werden.
3.  **Prüfen (Evaluation):** Der Code wandert Anzeige für Anzeige und Feld für Feld durch, wendet die obigen Spielregeln an und zählt die Punkte (`n_korrekt`) hoch.
4.  **Rechnen & Drucken:** Am Ende wird der Prozentsatz berechnet und die finale Tabelle ausgegeben.

## Phase 4 — Iteration A Auswertung

Hypothese-Cell *vor* der Iteration: welches Feld, welche Δ-Größe, warum?

## Phase 4 — Iteration B Auswertung

Hypothese-Cell *vor* der Iteration.

## Phase 4 — Synthese

Iterations-Tabelle (Baseline / A / B mit Hypothese, Aktion, Δ Gesamt, Δ schwächstes Feld, Diagnose) + Synthese-Antworten zu den drei Fragen aus dem Aufgabenblatt.

## Phase 6 — Vollständiger 7B-Run + 3B-Halluzinations-Klassen

Per-Field-Accuracy auf den 12 Hand-Gold-Anzeigen + Schema-Konformitäts-Check auf den restlichen Anzeigen ohne Gold. 3B-vs-7B-vs-Gold per `refnr` joinen, drei eigenständige Halluzinations-Klassen mit konkreten Beispielen identifizieren.